<a href="https://colab.research.google.com/github/davidrpugh/introduction-to-deep-learning/blob/master/notebooks/02d-building-an-image-classifier-with-pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building an Image Classifier with PyTorch

In [ ]:
%%bash

pip install --upgrade torchmetrics

In [ ]:
import pathlib


import pandas as pd
import torch
from torch import nn, optim, utils
import torchmetrics
import torchvision
import torchvision.transforms.v2 as T


# default linewidth is 80 characters
torch.set_printoptions(linewidth=120)


## Verifying availability of GPU(s)

In [ ]:
print(torch.__version__)

In [ ]:
%%bash

nvidia-smi

In [ ]:
print(torch.cuda.is_available())

In [ ]:
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [ ]:
print(DEVICE)

## Loading the Fashion MNIST dataset using TorchVision

[TorchVision](https://docs.pytorch.org/vision/stable/index.html) is a core PyTorch library for computer vision. Torchvision provides:

* Tools to download common datasets (e.g., MNIST, FashionMNIST).
* Pretrained models for vision tasks.
* Image transformations (crop, rotate, resize, etc.).

TorchVision is preinstalled on Google Colab and Kaggle making it easy to use in teaching and research.

### Fashion MNIST

The Fashion MNIST dataset has the same structure of the familiar MNIST dataset.

* 60,000 training images
* 10,000 test images
* Images are single channel (i.e., grayscale) images with 28 x 28 = 784 pixels.

### Image Preprocessing with Transforms

* TorchVision datasets accept a `transform` argument for preprocessing.
* Common transforms: scaling, normalization, cropping, etc.
* Use `Compose` to chain multiple transforms.
* `ToImage`: converts input to a Tensor image.
* `ToDtype`: converts to float32 and scales pixel values to [0.0, 1.0].

**Be sure to use version 2 of the TorchVision transforms (i.e., `torchvision.transforms.v2`)! Version 2 is much faster, has more transforms and features, and is backward-compatible with version 1.**

In [ ]:
DATA_DIR = pathlib.Path("./sample_data")


to_tensor = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
])


train_val_dataset = (
    torchvision.datasets
               .FashionMNIST(
                   DATA_DIR,
                   train=True,
                   download=True,
                   transform=to_tensor
               )
)

test_dataset = (
    torchvision.datasets
               .FashionMNIST(
                   DATA_DIR,
                   train=False,
                   download=True,
                   transform=to_tensor
               )
)

In [ ]:
%%bash

ls ./sample_data/FashionMNIST/raw

In [ ]:
X0, y0 = train_val_dataset[0]
print(X0.shape)
print(X0.dtype)

In [ ]:
train_val_dataset.classes[y0]

## Prepare the data

### Train/Val split

In [ ]:
_ = torch.manual_seed(42)

train_dataset, val_dataset = (
    utils.data
         .random_split(
             train_val_dataset,
             [55_000, 5_000]
         )
)

### Create the DataLoaders

In [ ]:
data_loader_kwargs = {
    "batch_size": 32,
    "num_workers": 2,            # load data in parallel using multiple workers
    "persistent_workers": True,  # keep workers around between epochs
    "pin_memory": True,          # avoid extra copy of data batches
    "prefetch_factor": 2,        # fetch multiple data batches in advance
}


train_data_loader = (
    utils.data
         .DataLoader(
             train_dataset,
             shuffle=True,
             **data_loader_kwargs
         )
)

val_data_loader = (
    utils.data
         .DataLoader(
             val_dataset,
             shuffle=False,
             **data_loader_kwargs
         )
)

test_data_loader = (
    utils.data
         .DataLoader(
             test_dataset,
             shuffle=False,
             **data_loader_kwargs
         )
)

### Wrapping our model in a custom module

In [ ]:
class MLPClassifier(nn.Module):

    def __init__(self, input_size, hidden_layer_sizes, n_classes):
        super().__init__()

        # create the hidden layers
        modules = nn.ModuleList([nn.Flatten()])
        for hidden_layer_size in hidden_layer_sizes:
            modules.append(nn.Linear(input_size, hidden_layer_size))
            modules.append(nn.ReLU())
            input_size = hidden_layer_size

        # define the output layer for the classifier
        modules.append(nn.Linear(input_size, n_classes))

        # create the MLP from the modules
        self.mlp = nn.Sequential(*modules)

    def forward(self, X):
        return self.mlp(X)



## Defining the training and evaluation loop

In [ ]:
def evaluate(model_fn, data_loader, metric):
    model_fn.eval()
    metric.reset()  # reset the metric at the beginning
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            y_pred = model_fn(X_batch)
            metric.update(y_pred, y_batch)  # update it at each iteration
    return metric.compute()  # compute the final result at the end


def train(
    model_fn,
    criterion,
    optimizer,
    metric,
    train_data_loader,
    val_data_loader,
    n_epochs,
    log_epochs=1,
    ):

    history = {
        "train_losses": [],
        "val_losses": [],
        "train_metrics": [],
        "val_metrics": [],
    }

    for epoch in range(n_epochs):
        total_train_loss = 0.0
        metric.reset()
        for i, (X_batch, y_batch) in enumerate(train_data_loader):
            model_fn.train()

            # move batches to device
            X_batch = X_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)

            # forward pass
            y_pred = model_fn(X_batch)
            train_loss = criterion(y_pred, y_batch)
            total_train_loss += train_loss.item()

            # backward pass
            train_loss.backward()

            # gradient descent step
            optimizer.step()
            optimizer.zero_grad()

            # update our metric
            metric.update(y_pred, y_batch)

        # comute the average (across batches!) training loss
        average_train_loss = total_train_loss / len(train_data_loader)
        history["train_losses"].append(average_train_loss)

        # compute the average (across batched!) validation loss
        with torch.no_grad():
            model_fn.eval()
            total_val_loss = 0.0
            for X_batch, y_batch in val_data_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                y_pred = model_fn(X_batch)
                val_loss = criterion(y_pred, y_batch)
                total_val_loss += val_loss.item()
            average_val_loss = total_val_loss / len(val_data_loader)
            history["val_losses"].append(average_val_loss)

        # compute the training metric after each epoch
        average_train_metric = (
            metric.compute()
                  .item()
        )
        history["train_metrics"].append(average_train_metric)

        # compute the validation metric after each epoch
        average_val_metric = (
            evaluate(
              model_fn,
              val_data_loader,
              metric,
            ).item()
        )
        history["val_metrics"].append(average_val_metric)

        if (epoch + 1) % log_epochs == 0:
            print(f"Epoch {epoch + 1}/{n_epochs}, "
                  f"train loss: {history['train_losses'][-1]:.4f}, "
                  f"val loss: {history['val_losses'][-1]:.4f}, "
                  f"train metric: {history['train_metrics'][-1]:.4f}, "
                  f"val metric: {history['val_metrics'][-1]:.4f}"
            )

    return history


## Putting everything together!

In [ ]:
_ = torch.manual_seed(42)

# define the model function
fashion_mnist_model_fn = MLPClassifier(
    input_size=28 * 28,
    hidden_layer_sizes=[256, 128],
    n_classes=10
)
fashion_mnist_model_fn = fashion_mnist_model_fn.to(DEVICE)

# select loss function
cross_entropy_loss = nn.CrossEntropyLoss()

# define the optimizer
sgd = optim.SGD(
    fashion_mnist_model_fn.parameters(),
    lr=1e-1
)

# select a metric
accuracy = (
    torchmetrics.Accuracy(
        task="multiclass",
        num_classes=10,
    ).to(DEVICE)
)


In [ ]:
history = train(
    model_fn=fashion_mnist_model_fn,
    criterion=cross_entropy_loss,
    optimizer=sgd,
    metric=accuracy,
    train_data_loader=train_data_loader,
    val_data_loader=val_data_loader,
    n_epochs=20,
    log_epochs=1
)

In [ ]:
history_df = pd.DataFrame.from_dict(
    history
)

_ = history_df.plot(grid=True)

## Predicting using the trained model

### Predicting class labels

In [ ]:
def predict(X, model_fn):
    model_fn.eval()
    with torch.no_grad():
        y_pred_logits = model_fn(X)
    class_indices = torch.argmax(y_pred_logits, dim=1)
    return class_indices

In [ ]:
X_new, y_new = next(iter(val_data_loader))

In [ ]:
X_new.device

In [ ]:
X_new = X_new.to(DEVICE)

In [ ]:
class_indices = predict(X_new, fashion_mnist_model_fn)
print(class_indices)

In [ ]:
class_labels = [train_val_dataset.classes[i] for i in class_indices]
print(class_labels)

### Predicting class probabilities

In [ ]:
def predict_proba(X, model_fn):
    model_fn.eval()
    with torch.no_grad():
        y_pred_logits = model_fn(X)
        y_pred_proba = torch.softmax(y_pred_logits, dim=1)
    return y_pred_proba


In [ ]:
class_probas = predict_proba(X_new, fashion_mnist_model_fn)
print(class_probas.round(decimals=3))

### Top-k predictions

In [ ]:
def predict_topk(X, model_fn, k=3):
    model_fn.eval()
    with torch.no_grad():
        y_pred_logits = model_fn(X)
        _, topk_class_indices = torch.topk(y_pred_logits, k=k, dim=1)
    return topk_class_indices


def predict_topk_proba(X, model_fn, k=3):
    model_fn.eval()
    with torch.no_grad():
        y_pred_logits = model_fn(X)
        topk_logits, topk_class_indices = torch.topk(y_pred_logits, k=k, dim=1)
        topk_probas = torch.softmax(topk_logits, dim=1)
    return topk_probas



In [ ]:
top3_class_indices = predict_topk(X_new, fashion_mnist_model_fn, k=3)
print(top3_class_indices)

In [ ]:
top3_class_labels = []
for class_indices in top3_class_indices:
    top3_class_labels.append(
        [train_val_dataset.classes[i] for i in class_indices]
    )
print(top3_class_labels)


In [ ]:
top3_probas = predict_topk_proba(X_new, fashion_mnist_model_fn, k=3)
print(top3_probas.round(decimals=3))